# jepa-as-a-tool: an open Meta model as the agent (Colab, H100)

**Muse Glimmer 30B** (open weights) plays the same tasks as Muse Spark, alone and with the **final
world model** (the one trained on three worlds in `train_and_evaluate.ipynb`). The question: can an
open model, on one GPU, use a world model as a tool?

Muse Spark on the same 50 tasks (seed 0, tasks 0–49):

| Tasks | Alone | With the world model | With the real physics |
|---|---|---|---|
| Tabletop (3D), pose | 64% | 92% | 98% |
| PushT (2D), rotate | 62% | 78% | 96% |

The model runs on this GPU behind vLLM (an OpenAI-compatible server); the agent talks to it exactly
like it talks to the Meta API. V-JEPA 2 and the predictor share the GPU.

**Before running:**
1. Runtime → Change runtime type → **H100 GPU** (80 GB). A 40 GB A100 is too small: the weights
   alone take 33 GB, nothing is left for the conversations.
2. Hugging Face token (read access): huggingface.co → Settings → Access Tokens → Create new token →
   type **Read** → copy it. Open the model page (`MODEL` below) and accept its license if asked.
3. Colab: the **key icon** (Secrets) on the left → Add new secret → name `HF_TOKEN`, value: the
   token → turn on **Notebook access**. Never paste the token in a cell.
4. Upload the main run's **`results.zip`** to Google Drive, in a folder `jat/` (My Drive → jat).
5. Runtime → **Run all**, and keep this tab open (see below).

Results are written to Drive (`My Drive/jat/open-model-final/`) as each episode ends: if the runtime
disconnects, run all again and finished episodes are skipped.

## 0. GPU, Drive, Hugging Face token

In [ ]:
import os
import sys
from pathlib import Path

from google.colab import drive, userdata

!nvidia-smi --query-gpu=name,memory.total --format=csv
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/jat")
OUT = DRIVE / "open-model-final"
OUT.mkdir(parents=True, exist_ok=True)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF token loaded:", bool(os.environ["HF_TOKEN"]), "| results go to", OUT)

## 1. Install: vLLM in its own environment, the project in Colab's

vLLM pins its own PyTorch: in a separate environment it cannot break the project's. ~5 min.

In [ ]:
!pip install -q uv
!uv venv -q /content/vllm-env --python 3.12
!uv pip install -q --python /content/vllm-env/bin/python vllm
!/content/vllm-env/bin/python -c "import vllm; print('vLLM', vllm.__version__)"

WORK = "/content"
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!git log --oneline -1

## 2. Rendering check and the final world model (from Drive)

In [ ]:
import subprocess
import sys

!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

CHECK = "from jat.worlds import get_world; w = get_world('tabletop'); f = w.render(w.sample_task('pose', 0, 0).start); print('render ok', f.shape, f.mean().round(1))"


def renders(gl):
    env = {**os.environ, "MUJOCO_GL": gl}
    out = subprocess.run([sys.executable, "-c", CHECK], env=env, capture_output=True, text=True)
    print(gl, "->", (out.stdout or out.stderr).strip().splitlines()[-1])
    return out.returncode == 0


if renders("egl"):
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq install -y libosmesa6 > /dev/null
    assert renders("osmesa"), "MuJoCo cannot render here"
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
import zipfile

INPUTS = Path("/content/inputs")
WM = INPUTS / "runs" / "wm"
if not (WM / "predictor.pt").exists():
    with zipfile.ZipFile(DRIVE / "results.zip") as archive:
        archive.extractall(INPUTS, [n for n in archive.namelist() if n.startswith("runs/wm/")])
print("final world model:", sorted(p.name for p in WM.iterdir()))

## 3. Start Muse Glimmer behind vLLM

Downloads the weights the first time (~60 GB, ~5–15 min), then loads them. Memory decides the
weights are cut to 8 bits (fp8, 33 GB in memory), so the GPU needs 80 GB: on a 40 GB one
nothing is left for the conversations. ~5 GB stay free for V-JEPA 2 and the predictor.
The server's log: `/content/vllm.log`.

In [ ]:
import json
import time
import urllib.request

import torch

MODEL = "meta-models/Muse-Glimmer-30B"
TOOL_PARSER = "muse_glimmer"
MAX_LEN = 32768  # the longest Muse Spark episode used 24k tokens
WORKERS = 12  # episodes played at once
PORT = 8000

total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert total_gb > 70, f"{total_gb:.0f} GB GPU: too small, take an H100 (80 GB)"
QUANTS = ["fp8"]  # this vLLM has no 4-bit bitsandbytes
UTIL = round((total_gb - 5) / total_gb, 2)
print(f"{total_gb:.0f} GB GPU: trying {QUANTS}, vLLM gets {UTIL:.0%} of the memory")


def ready():
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5)
        return True
    except OSError:
        return False


def start(quant):
    cmd = [
        "/content/vllm-env/bin/vllm", "serve", MODEL, "--served-model-name", "glimmer",
        "--port", str(PORT), "--max-model-len", str(MAX_LEN),
        "--gpu-memory-utilization", str(UTIL), "--max-num-seqs", str(2 * WORKERS),
        "--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER, "--quantization", quant,
    ]  # fmt: skip
    log = open("/content/vllm.log", "a")  # noqa: SIM115 (open as long as the server runs)
    log.write(f"\n$ {' '.join(cmd)}\n")
    log.flush()
    server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    started = time.time()
    while server.poll() is None and not ready():
        time.sleep(15)
        print(f"\r  {quant}: loading, {(time.time() - started) / 60:.0f} min", end="")
    print()
    if server.poll() is None:
        return server
    print(f"  {quant} failed:")
    !tail -n 25 /content/vllm.log
    return None


server = None if ready() else next((s for s in map(start, QUANTS) if s is not None), None)
assert ready(), "vLLM did not start: send me the end of /content/vllm.log"
print("Muse Glimmer is up")

## 4. Smoke test: does it call a tool?

One request with one tool. The answer must contain a tool call, not only text.

In [ ]:
request = {
    "model": "glimmer",
    "messages": [{"role": "user", "content": "Push the box 50 px to the right, starting at (100, 200)."}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "push",
            "description": "Push in a straight line.",
            "parameters": {
                "type": "object",
                "properties": {"start": {"type": "array", "items": {"type": "number"}},
                               "end": {"type": "array", "items": {"type": "number"}}},
                "required": ["start", "end"],
            },
        },
    }],
}  # fmt: skip
http = urllib.request.Request(
    f"http://localhost:{PORT}/v1/chat/completions",
    data=json.dumps(request).encode(),
    headers={"Content-Type": "application/json"},
)
started = time.time()
reply = json.loads(urllib.request.urlopen(http, timeout=600).read())
message = reply["choices"][0]["message"]
print(f"{time.time() - started:.0f} s, {reply['usage']['completion_tokens']} tokens")
print("text:", (message.get("content") or "")[:300])
print("tool calls:", message.get("tool_calls"))
assert message.get("tool_calls"), "no tool call: send me this output"

## 5. One real episode

The agent on one tabletop task, with the world model. Look at the last lines: it should call
`simulate`, then `act`, then `done`. Also note the time per episode.

In [ ]:
os.environ["LOCAL_API_KEY"] = "none"
os.environ["LOCAL_BASE_URL"] = f"http://localhost:{PORT}/v1"
os.environ["LOCAL_MODEL"] = "glimmer"

!jat eval --out {OUT}/smoke --world tabletop --families pose --n 1 --settings wm_tool --provider local --workers 1 --wm {WM}

## 6. The experiment: 50 tabletop tasks, alone and with the world model

100 episodes, `WORKERS` at a time. Progress prints one line per episode; the full log is in
Drive (`open-model-final/eval.log`). If the runtime stops, run this cell again: it continues.

**Background:** on Colab Pro, closing the tab or leaving it idle a long time can stop the runtime.
Keep the tab open and the computer awake (Pro+ can run in the background). Everything finished is
already in Drive.

In [ ]:
RESULTS = OUT / "tabletop"
!jat eval --out {RESULTS} --world tabletop --families pose --n 50 --settings no_tool,wm_tool --provider local --workers {WORKERS} --wm {WM} 2>&1 | tee -a {OUT}/eval.log

## 7. Results

Muse Spark on the same tasks: alone 64%, with the world model 92%, with the real physics 98%.

In [ ]:
!jat report {RESULTS} | tee {RESULTS}/report.txt

## 8. If there is time: the same on PushT (2D)

Muse Spark on these 50 tasks: alone 62%, with the world model 78%, with the real physics 96%.

In [ ]:
PUSHT = OUT / "pusht"
!jat eval --out {PUSHT} --world pusht --families rotate --n 50 --settings no_tool,wm_tool --provider local --workers {WORKERS} --wm {WM} 2>&1 | tee -a {OUT}/eval.log
!jat report {PUSHT} | tee {PUSHT}/report.txt

## Send back

Everything is in Drive: `My Drive/jat/open-model-final/` (results, `eval.log`). The cell below
makes `open-model-final.zip` next to it: send that.

In [ ]:
!cd {DRIVE} && zip -qr open-model-final.zip open-model-final && ls -lh open-model-final.zip